# CNN Encoder-Decoder
This code was originally purpoused for turning images took in the day to nightime images, but the dataset was unpaired and I then had to repurpouse. It now uses the edge to shoes dataset (Edges2Shoes on kaggle). This is a simple CNN that turns images to images using an encoder-decoder architecture. The training code is below.

After running, run the cleanup script. If you have not ran the cleaning script and downloaded all the output, (like I did on my first run) unzip the output (it should be 1.32 GB) then delete everything except for edges-to-shoes.keras.  After that, use the file as an input (in the kaggle IDE, this is what you do. In other IDEs, feel free to edit file paths). Then run the test code.

In [ ]:
# Training code

import tensorflow as tf
import cv2
import os
from tqdm import tqdm
import shutil

def combined_loss(y_true, y_pred):
        ssim_loss = 1.0 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))
        l1_loss = tf.reduce_mean(tf.abs(y_true - y_pred))
        return l1_loss + (0.1 * ssim_loss)

def split_and_normalize(batch):
    batch = batch / 255.0
    edges = batch[:, :, :256, :] 
    shoes = batch[:, :, 256:, :]
    return (edges, shoes)

img_height, img_width = 256, 256
batch_size = 8

raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    "/kaggle/input/datasets/balraj98/edges2shoes-dataset/train",
    label_mode=None,
    batch_size=8,
    image_size=(256, 512),
    shuffle=True,         
    seed=123
)

raw_val_ds = tf.keras.utils.image_dataset_from_directory(
    "/kaggle/input/datasets/balraj98/edges2shoes-dataset/val",
    label_mode=None,
    batch_size=8,
    image_size=(256, 512),
    shuffle=False
)

train_ds = raw_train_ds.map(split_and_normalize).prefetch(tf.data.AUTOTUNE)
val_ds = raw_val_ds.map(split_and_normalize).prefetch(tf.data.AUTOTUNE)

inputs = tf.keras.layers.Input(shape=(256,256,3))

c1 = tf.keras.layers.Conv2D(64, 3, padding='same')(inputs)
b1 = tf.keras.layers.BatchNormalization()(c1)
c1 = tf.keras.layers.LeakyReLU()(b1)

cu2 = tf.keras.layers.Conv2D(64, 3, padding='same')(c1)
b2 = tf.keras.layers.BatchNormalization()(cu2)
c2 = tf.keras.layers.LeakyReLU()(b2)

p1 = tf.keras.layers.MaxPooling2D()(c2)

c3 = tf.keras.layers.Conv2D(128, 3, padding='same')(p1)
b3 = tf.keras.layers.BatchNormalization()(c3)
c3 = tf.keras.layers.LeakyReLU()(b3)

cu4 = tf.keras.layers.Conv2D(128, 3, padding='same')(c3)
b4 = tf.keras.layers.BatchNormalization()(cu4)
c4 = tf.keras.layers.LeakyReLU()(b4)

p2 = tf.keras.layers.MaxPooling2D()(c4)

bo1 = tf.keras.layers.Conv2D(256, 3, padding='same')(p2)
b5 = tf.keras.layers.BatchNormalization()(bo1)
bo1 = tf.keras.layers.LeakyReLU()(b5)

bo2 = tf.keras.layers.Conv2D(256, 3, padding='same')(bo1)
b6 = tf.keras.layers.BatchNormalization()(bo2)
bo2 = tf.keras.layers.LeakyReLU()(b6)

u1 = tf.keras.layers.UpSampling2D()(bo2)
u2 = tf.keras.layers.Concatenate()([u1, c4])

c5 = tf.keras.layers.Conv2D(128, 3, padding='same')(u2)
b7 = tf.keras.layers.BatchNormalization()(c5)
c5 = tf.keras.layers.LeakyReLU()(b7)

c6 = tf.keras.layers.Conv2D(128, 3, padding='same')(c5)
b8 = tf.keras.layers.BatchNormalization()(c6)
c6 = tf.keras.layers.LeakyReLU()(b8)

u3 = tf.keras.layers.UpSampling2D()(c6)
u4 = tf.keras.layers.Concatenate()([u3, c2])

c7 = tf.keras.layers.Conv2D(64, 3, padding='same')(u4)
b9 = tf.keras.layers.BatchNormalization()(c7)
c7 = tf.keras.layers.LeakyReLU()(b9)

c8 = tf.keras.layers.Conv2D(64, 3, padding='same')(c7)
b10 = tf.keras.layers.BatchNormalization()(c8)
c8 = tf.keras.layers.LeakyReLU()(b10)

outputs = tf.keras.layers.Conv2D(3, 1, activation='sigmoid', padding='same', dtype='float32')(c8)

model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002), loss=combined_loss)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath="/kaggle/working/edges-to-shoes.keras", 
    save_best_only=True,         
    monitor="val_loss"
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    patience=10, 
    monitor="val_loss",
    restore_best_weights=True  
)

model.fit(train_ds, epochs=20, callbacks=[early_stopping, checkpoint], validation_data=val_ds)

In [ ]:
# Testing if dataset exists
import os
os.listdir("/kaggle/input/datasets/balraj98/edges2shoes-dataset")

In [ ]:
# Cleanup the output code
# Run directly after training code
import shutil
import os

train_X = "/kaggle/working/train_X"
train_y = "/kaggle/working/train_y"

val_X = "/kaggle/working/val_X"
val_y = "/kaggle/working/val_y"
paths = [train_X, train_y, val_X, val_y]

for path in paths:
    if os.path.exists(path):
        shutil.rmtree(path)
    else:
        print(f'{path} does not exist. Have you ran the training code?')

In [ ]:
##### import tensorflow as tf
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2

tf.keras.config.enable_unsafe_deserialization()

model = tf.keras.models.load_model("/kaggle/input/models/zacharypotishko/v9/keras/default/1/edges-to-shoes-3.keras", compile=False)

img = cv2.imread("/kaggle/input/datasets/balraj98/edges2shoes-dataset/train/10135_AB.jpg")
    
h, w, _ = img.shape
midpoint = w // 2
    
right = img[:, :midpoint]
left = img[:, midpoint:]
right = cv2.cvtColor(right, cv2.COLOR_BGR2RGB)
left = cv2.cvtColor(left, cv2.COLOR_BGR2RGB)

resized_img = cv2.resize(right, (256, 256))
left_resized = cv2.resize(left, (256, 256))

print("Path loading completed")

input_tensor = resized_img[None, :, :, :] / 255
output = model.predict(input_tensor)

print("Image Loaded and processed")

fig, axes = plt.subplots(2, 2, figsize=(10, 5))
axes[0, 0].imshow(resized_img)
axes[0, 1].imshow(output.squeeze())
axes[1, 0].imshow(left_resized)
axes[1, 1].axis('off')
plt.show()

# MLP with Embeddings
This code uses a custom embedding look up table that then has a simple classification model after it. This model aims to perdition the p150 of a molecule based on its SMILES and other factors.

In [4]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Input, Embedding, Flatten, Dense, concatenate, Normalization, BatchNormalization
from tensorflow.keras.models import Model
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/yanmaksi/big-molecules-smiles-dataset/SMILES_Big_Data_Set.csv")

df = df.dropna(subset=['pIC50', 'SMILES', 'logP', 'num_atoms'])

df = df[df['pIC50'].apply(lambda x: isinstance(x, (int, float)))]

y = df['pIC50'].values
X_smiles = df['SMILES'].values 
X_numeric = df[['num_atoms', 'logP']].values

vectorize_layer = TextVectorization(
    standardize='lower',
    max_tokens=128,
    split="character",
    output_mode='int',
    output_sequence_length=100)

vectorize_layer.adapt(X_smiles)

embedding_dim=16

smiles_input = Input(shape=(1,), dtype=tf.string, name="smiles_raw_text")
vectorized = vectorize_layer(smiles_input)
embedded = Embedding(input_dim=128, output_dim=embedding_dim, name="embedding")(vectorized)
pooled_text = Flatten()(embedded)

numeric_input = Input(shape=(2,), name="numeric_features")
normalizer = Normalization(axis=-1)
normalizer.adapt(X_numeric)  
normalized_numeric = normalizer(numeric_input)

combined = concatenate([pooled_text, normalized_numeric])
x = BatchNormalization()(combined)
x = Dense(256, activation='relu')(x)
x = Dense(128, activation='relu')(x)
output = Dense(1)(x)

model = tf.keras.Model(inputs=[smiles_input, numeric_input], outputs=output)

model.compile(optimizer='adam',
              loss='mse',
              metrics=['mae'])

model.fit(
    x=[X_smiles, X_numeric],
    y=y,
    epochs=15,
    batch_size=32
)

model.save("SMILES-P150.keras")

Epoch 1/15
470/470 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 2.2506 - mae: 0.7626
Epoch 2/15
470/470 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.6616 - mae: 0.3700
Epoch 3/15
470/470 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.3890 - mae: 0.2861
Epoch 4/15
470/470 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.3070 - mae: 0.2501
Epoch 5/15
470/470 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.2641 - mae: 0.2277
Epoch 6/15
470/470 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1853 - mae: 0.1928
Epoch 7/15
470/470 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1893 - mae: 0.1931
Epoch 8/15
470/470 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1773 - mae: 0.1788
Epoch 9/15
470/470 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.2254 - mae: 0.1931
Epoch 10/15
470/470 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1696 - mae: 0.1701
Epoch 11/15
470/470 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1490 - mae: 0.1618
Epoch 12/15
470/470 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1379 - mae: 0.1515
Epoch 13/15
470/470 ━━━━━

In [7]:
import tensorflow as tf
import numpy as np
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/yanmaksi/big-molecules-smiles-dataset/SMILES_Big_Data_Set.csv")
std = df['pIC50'].std()
mean = df['pIC50'].mean()

model = tf.keras.models.load_model('/kaggle/working/SMILES-P150.keras')

test_smiles = tf.constant([[" CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5"]], dtype=tf.string)
test_numeric = tf.constant([[68, 3.51]], dtype=tf.float32)
prediction = model.predict([test_smiles, test_numeric])

prediction = (prediction * std) + mean
print(f"p150: {prediction[0][0]:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
p150: 12.8639
